# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [9]:
# Load the libraries as required.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import shap
import pickle 

In [35]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = pd.read_csv("C:/Users/manke/OneDrive/UofT DSI/MLOPs/05_src/data/fires/forestfires.csv", header = 0, names = columns)
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [22]:
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [40]:
numeric = ['coord_x', 'coord_y','ffmc','dmc','dc','isi','temp','rh','wind','rain']
categorical = ['month','day']

preproc1 = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat',OneHotEncoder(drop='first',sparse_output=False),categorical)
],remainder='drop')

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
preproc2 = ColumnTransformer([
    ('num', Pipeline([
        ('power', PowerTransformer(method='yeo-johnson')),  # Non-linear
        ('scaler', StandardScaler())
    ]), numeric),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical)
], remainder='drop')

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [42]:
# Pipeline A = preproc1 + baseline
pipeA = Pipeline([
    ('preprocessing',preproc1),
    ('regressor',Ridge(random_state=42))
])

In [43]:
sorted(pipeA.get_params().keys())

['memory',
 'preprocessing',
 'preprocessing__cat',
 'preprocessing__cat__categories',
 'preprocessing__cat__drop',
 'preprocessing__cat__dtype',
 'preprocessing__cat__handle_unknown',
 'preprocessing__cat__max_categories',
 'preprocessing__cat__min_frequency',
 'preprocessing__cat__sparse',
 'preprocessing__cat__sparse_output',
 'preprocessing__n_jobs',
 'preprocessing__num',
 'preprocessing__num__copy',
 'preprocessing__num__with_mean',
 'preprocessing__num__with_std',
 'preprocessing__remainder',
 'preprocessing__sparse_threshold',
 'preprocessing__transformer_weights',
 'preprocessing__transformers',
 'preprocessing__verbose',
 'preprocessing__verbose_feature_names_out',
 'regressor',
 'regressor__alpha',
 'regressor__copy_X',
 'regressor__fit_intercept',
 'regressor__max_iter',
 'regressor__positive',
 'regressor__random_state',
 'regressor__solver',
 'regressor__tol',
 'steps',
 'verbose']

In [27]:
# Pipeline B = preproc2 + baseline
pipeB = Pipeline([
    ('preprocessing',preproc2),
    ('regressor',Ridge(random_state=42))
])

In [44]:
# Pipeline C = preproc1 + advanced model
pipeC = Pipeline([
    ('preprocessing',preproc1),
    ('regressor',RandomForestRegressor(random_state=42))
])

In [29]:
# Pipeline D = preproc2 + advanced model
pipeD = Pipeline([
    ('preprocessing',preproc2),
    ('regressor',RandomForestRegressor(random_state=42))
])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [45]:
p_grid_A = {'regressor__alpha' : [0.1, 1.0, 5.0, 10.0]}

grid_A = GridSearchCV(pipeA, p_grid_A, cv=5, scoring='neg_mean_absolute_error',n_jobs=-1)
grid_A.fit(X_train, y_train)

One or more of the test scores are non-finite: [nan nan nan nan]


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge(random_state=42))]),
             n_jobs=-1, param_grid={'regressor__alpha': [0.1, 1.0, 5.0, 10.0]},
             scoring='neg_mean_absolute_error')

In [46]:
p_grid_B = {'regressor__alpha' : [0.1, 1.0, 5.0, 10.0]}

grid_B = GridSearchCV(pipeB, p_grid_B, cv=5, scoring='neg_mean_absolute_error',n_jobs=-1)
grid_B.fit(X_train, y_train)

One or more of the test scores are non-finite: [nan nan nan nan]


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('power',
                                                                                          PowerTransformer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge(random_state=42))]),
             n_jobs=-1, param_grid={'regressor__alpha': [0.1, 1.0, 5.0, 10.0]},
             scoring='neg_mean_absolute_error')

In [48]:
p_grid_C = {'regressor__n_estimators': [50, 100, 150, 200]}

grid_C = GridSearchCV(pipeC, p_grid_C, cv=5, scoring='neg_mean_absolute_error',n_jobs=-1)
grid_C.fit(X_train, y_train)

One or more of the test scores are non-finite: [nan nan nan nan]


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'regressor__n_estimators': [50, 100, 150, 200]},
             scoring='neg_mean_absolute_error')

In [49]:
p_grid_D = {'regressor__n_estimators': [50, 100, 150, 200]}

grid_D = GridSearchCV(pipeD, p_grid_D, cv=5, scoring='neg_mean_absolute_error',n_jobs=-1)
grid_D.fit(X_train, y_train)

One or more of the test scores are non-finite: [nan nan nan nan]


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('power',
                                                                                          PowerTransformer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'regressor__n_estimators': [50, 100, 150, 200]},
             scoring='neg_mean_absolute_error')

In [50]:
#Perfomance Testing, save results to dict
best_pipe = {
    'Pipeline A': grid_A.best_estimator_,
    'Pipeline B': grid_B.best_estimator_,
    'Pipeline C': grid_C.best_estimator_,
    'Pipeline D': grid_D.best_estimator_
}

results = {}
for name, pipeline in best_pipe.items():

    y_pred = pipeline.predict(X_test)

    #metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {'Mean Abs Error' : mae,'Root Mean Error' : rmse, 'R Squared' : r2}

    print(f'{name} Results: MAE {mae}, RMSE {rmse}, R Squared {r2}')

Pipeline A Results: MAE 24.402788497484867, RMSE 107.7873422582519, R Squared 0.014392127010644185
Pipeline B Results: MAE 24.612188622679145, RMSE 107.56425673090796, R Squared 0.01846769492021827
Pipeline C Results: MAE 26.658737742673996, RMSE 109.25235236727603, R Squared -0.012582063970633994
Pipeline D Results: MAE 26.54114986263736, RMSE 110.37931884755238, R Squared -0.03357990299483027


# Evaluate

+ Which model has the best performance?

In [51]:
#flip rows and cols of df
results_df = pd.DataFrame(results).T 

#Find Best Model
best_model_name = results_df['Mean Abs Error'].idxmin()
best_model = best_pipe[best_model_name]

print(f'Best Model = {best_model_name}')

Best Model = Pipeline A


# Export

+ Save the best performing model to a pickle file.

In [52]:
with open('best_fires_model.pkl', 'wb') as f:
    pickle.dump(best_model,f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
X_train_transformed = best_model.named_steps['preprocessing'].transform(X_train)
X_test_transformed = best_model.named_steps['preprocessing'].transform(X_test)

#explainer
explainer = shap.Explainer(best_model.named_steps['regressor'], X_train_transformed)
shap_values = explainer(X_test_transformed)

# Get feature names
num_features = numeric
cat_features_transformed = best_model.named_steps['preprocessing'].named_transformers_['cat'].get_feature_names_out(categorical)
all_feature_names = num_features + list(cat_features_transformed)

In [68]:
#local
sample_idx = 0
print(f"Actual area: {y_test.iloc[sample_idx]:.4f}")
print(f"Predicted area: {best_model.predict(X_test.iloc[[sample_idx]])[0]}")

#most important
sample_shap = shap_values[sample_idx].values
feature_importance = list(zip(all_feature_names, sample_shap))
feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

print("Most important features for this prediction:")
for i, (feature, importance) in enumerate(feature_importance[:5]):
    print(f"{i+1}. {feature}: {importance:.4f}")

Actual area: 0.0000
Predicted area: 34.61933273880767
Most important features for this prediction:
1. dc: 34.8895
2. month_may: 34.5307
3. dmc: -15.6302
4. month_sep: -14.9469
5. month_aug: -9.1363


In [69]:
#global
mean_shap = np.mean(np.abs(shap_values.values), axis=0)
global_importance = list(zip(all_feature_names, mean_shap))
global_importance.sort(key=lambda x: x[1], reverse=True)

print("Most important features globally:")
for i, (feature, importance) in enumerate(global_importance[:5]):
    print(f"{i+1}. {feature}: {importance:.4f}")

print("\nLeast important features globally:")
for i, (feature, importance) in enumerate(global_importance[-5:]):
    print(f"{i+1}. {feature}: {importance:.4f}")


Most important features globally:
1. month_sep: 16.8398
2. dc: 12.3831
3. month_aug: 12.1427
4. dmc: 7.6868
5. coord_x: 4.9628

Least important features globally:
1. ffmc: 0.2241
2. rain: 0.2184
3. month_jan: 0.1792
4. month_jun: 0.1001
5. month_nov: 0.0000


I would remove the least important features if there were any I had to remove. Intuitively January is in the winter months and thus the environment is less condusive to fires, and its reflected in the results. Rain is also in oppositions to spreading fires, so it intuitively matches up with the results. Lastly, we'd remove ffmc. 

The way to compare would be to create a new model excluding these features, and then analyzing the results of the two models side by side for differences.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.